# Mixture of experts: a hands-on tour

You will run **four models on your own data**, always with the same backbone, and look inside two of them:

| model | idea |
|---|---|
| `cnn` | baseline: backbone + plain dense head |
| `moe` | a gate decides how much to trust each expert (every expert sees the whole feature vector) |
| `pmoe` | every expert has its **own router** and only looks at the `l` patches of the feature map it likes best |
| `mge` | experts attached at **different depths** of the backbone, with a penalty that pushes them to disagree |

All code lives in `utils/moe_project/` (one job per file, see its `README.md`).

**Colab / VS Code note:** the first cell clones or pulls the GitHub repo. A fresh Colab VM only sees
`utils/moe_project/` after you have **committed and pushed it**.

Set `QUICK = True` below for a smoke run (a few batches, a few epochs). Set it to `False` for a real run.

## 0. Catching up
These five cells are copied unchanged from `MoE_approach.ipynb`: clone/pull the repo, imports, rebuild the
split folders on a fresh VM, read the labels, build the `tf.data` pipelines (`train`, `val`, `test`).

In [ ]:
import os, sys, subprocess
import warnings
warnings.filterwarnings('ignore')

REPO_URL = 'https://github.com/durak-dev/ImageClassification.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab starts with an empty VM: clone the repo (code AND data live in it).
    # On later runs the folder already exists, so just pull the newest commits.
    ROOT = '/content/ImageClassification'
    if not os.path.exists(ROOT):
        subprocess.run(['git', 'clone', REPO_URL, ROOT], check=True)
    else:
        subprocess.run(['git', '-C', ROOT, 'pull'], check=True)
else:
    # Local: the notebook sits in the project root, but also work if it was
    # started from a subfolder.
    cwd = os.path.abspath(os.getcwd())
    ROOT = cwd if os.path.isdir(os.path.join(cwd, 'utils')) else os.path.dirname(cwd)

os.chdir(ROOT)
sys.path.insert(0, ROOT)          # so `import utils.funcs` finds OUR utils folder

DATA_DIR = os.path.join(ROOT, 'data')
ORIG_DIR = os.path.join(DATA_DIR, 'original')            # raw images/ + labels/ (placed manually, never modified)
MANIFEST_PATH = os.path.join(DATA_DIR, 'split_manifest.csv')   # which file belongs to which split
SEED = 42
print('Running on:', 'Google Colab' if IN_COLAB else 'local machine')
print('Project root:', ROOT)

In [ ]:
import importlib
import utils.funcs
importlib.reload(utils.funcs)      # pick up edits to funcs.py without restarting
from utils.funcs import *
from utils import config, github_sync
from utils.github_sync import save_csv, load_csv, push_files
from utils.MoE import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Recreate data/{split}/ with the corrected images + labels if they are missing (fresh Colab VM).
# Needs split_manifest.csv, data/original/ and the two checkpoint files from the cloned repo.
if not all(os.path.exists(os.path.join(split_dir(DATA_DIR, s, 'labels'), CORRECTED_LABELS_FILE)) for s in SPLITS):
    materialize_splits(ORIG_DIR, DATA_DIR, pd.read_csv(MANIFEST_PATH))
    rebuild_corrected(pd.read_csv(os.path.join(ROOT, 'checkpoints', 'df_fixed.csv')), DATA_DIR,
                      labels_corrected=pd.read_csv(os.path.join(ROOT, 'checkpoints', 'labels_corrected.csv')))

In [ ]:
dfs = {s: pd.read_csv(os.path.join(split_dir(DATA_DIR, s, "labels"), CORRECTED_LABELS_FILE)) for s in ["train", "val", "test"]}
classes = sorted(dfs["train"]["class_id"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

In [ ]:
SIZE, BATCH, AUTOTUNE = 64, 128, tf.data.AUTOTUNE

def load(path, y):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, (SIZE, SIZE)), y

def make_ds(split, shuffle):
    df = dfs[split]
    paths = (f"{DATA_DIR}/{split}/images_corrected/" + df["filename"].astype(str)).values
    labels = df["class_id"].map(class_to_idx)
    assert labels.notna().all(), f"{split} has labels not present in train"
    ds = tf.data.Dataset.from_tensor_slices((paths, labels.values.astype("int32")))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED)
    return ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train, val, test = make_ds("train", True), make_ds("val", False), make_ds("test", False)
print(len(classes), "classes")
print(dfs["train"]["filename"].iloc[0], os.path.exists(f"{DATA_DIR}/train/images_corrected/" + str(dfs["train"]["filename"].iloc[0])))
for x, y in train.take(1):
    print(x.shape, x.dtype, float(tf.reduce_min(x)), float(tf.reduce_max(x)))

## 1. Imports for this tutorial
`MoE_approach.ipynb` defines its own function called `run`, and `utils.funcs` has its own `plot_history`.
To avoid mixing them up, everything from `moe_project` is used through its module name
(`moe_train.run`, `plotting.plot_history`, `models.build_pmoe`, ...).

The reload loop lets you edit a file in `utils/moe_project/` and re-run this cell without restarting the kernel.

In [ ]:
import importlib
from utils.moe_project import config as moe_config          # (utils.config is the GitHub one, not this)
from utils.moe_project import backbone, moe, pmoe, mge, callbacks, models, train as moe_train, plotting

# order matters: a module is reloaded after the modules it imports from
for module in (moe_config, backbone, moe, pmoe, mge, callbacks, models, moe_train, plotting):
    importlib.reload(module)

## 2. Settings
`train` and `val` come from the catch-up cells. `val` is not shuffled, which `MacroF1` relies on.

In [ ]:
QUICK = True                  # True: tiny smoke run. False: full data and the epochs from config.py
N_CLASSES = len(classes)      # `classes` comes from the catch-up cells; SIZE is set there too

if QUICK:
    train_ds = train.take(30)     # 30 batches
    val_ds = val.take(10)         # 10 batches, still not shuffled
    EPOCHS = 3
    STAGE_EPOCHS = 1
    SEEDS = (0,)
else:
    train_ds, val_ds = train, val
    EPOCHS = moe_config.EPOCHS
    STAGE_EPOCHS = moe_config.MGE_STAGE_EPOCHS
    SEEDS = moe_config.SEEDS

LOG_DIR = 'history_logs/tutorial'     # per-epoch CSVs of these runs; kept apart from your real logs
print(f'SIZE={SIZE}  classes={N_CLASSES}  epochs={EPOCHS}  quick={QUICK}')

## 3. What is a "patch"?
The backbone halves the image three times. **Before** global pooling it holds a grid of feature vectors,
one per position. pMoE calls each position a patch. MGE instead takes three maps from three depths.

In [ ]:
inp = tf.keras.Input((SIZE, SIZE, 3))
fmap = backbone.backbone_feature_map(inp)          # what pMoE reads
stages = backbone.backbone_multi_depth(inp)        # what the MGE experts read

print('feature map before pooling:', fmap.shape, '-> n =', backbone.n_patches(SIZE), 'patches of 256 numbers')
print('MGE depth maps:', [tuple(s.shape[1:]) for s in stages])

## 4. Build all four and compare sizes (no training yet)
Same builder style for all: `build_xxx(size, n_classes, ...)`. Look at the parameter counts:
the comparison is only fair if no model wins just because it is bigger.

In [ ]:
tf.keras.backend.clear_session()
demo = {
    'cnn':  models.build_cnn(SIZE, N_CLASSES, n_experts=4, hidden=128),
    'moe':  models.build_moe(SIZE, N_CLASSES, n_experts=4, hidden=128),
    'pmoe': models.build_pmoe(SIZE, N_CLASSES, n_experts=4, l=8),
    'mge':  models.build_mge(SIZE, N_CLASSES),
}
for name, m in demo.items():
    print(f'{name:5s} {m.count_params():>10,} parameters   outputs: {m.output_names}')

The `mge` model has **several outputs**: the average of the experts (`ensemble`, only for measuring) and one output per expert.
Each expert is trained with its own cross-entropy. `moe_train.run` hands the label to every output for you.

## 5. Baseline CNN and plain MoE
`moe_train.run(name, seed, train_ds, val_ds, ...)` builds, trains (early stopping, val macro-F1 each epoch,
CSV log) and returns the Keras `History`. The trained model is `history.model`.

In [ ]:
hist_cnn = moe_train.run('cnn', seed=0, train_ds=train_ds, val_ds=val_ds, size=SIZE, n_classes=N_CLASSES,
                         epochs=EPOCHS, log_dir=LOG_DIR)
plotting.plot_history(hist_cnn)

In [ ]:
hist_moe = moe_train.run('moe', seed=0, train_ds=train_ds, val_ds=val_ds, size=SIZE, n_classes=N_CLASSES,
                         epochs=EPOCHS, log_dir=LOG_DIR, n_experts=4)
plotting.plot_history(hist_moe)       # the "usage [...] wins [...]" lines above show how the gate shares the work

## 6. pMoE: which patches does each expert pick?
Extra arguments of `run` go to the builder, so `n_experts=` and `l=` below are arguments of `build_pmoe`.
`l` = patches kept per expert (must not exceed the number of patches printed in section 3).

In [ ]:
hist_pmoe = moe_train.run('pmoe', seed=0, train_ds=train_ds, val_ds=val_ds, size=SIZE, n_classes=N_CLASSES,
                          epochs=EPOCHS, log_dir=LOG_DIR, n_experts=4, l=8)
plotting.plot_history(hist_pmoe)

In [ ]:
images, labels = next(iter(val_ds))
top_idx, gates, grid = pmoe.expert_choices(hist_pmoe.model, images[:4])
print('top_idx', top_idx.shape, '= (images, experts, l patches);  patch grid', grid)
print('gate values of expert 0 on image 0:', gates[0, 0].round(3), ' (they sum to', gates[0, 0].sum().round(3), ')')

for i in range(3):                                   # one row per image, one panel per expert
    plotting.show_expert_patches(images[i].numpy(), top_idx[i], grid)

Do the experts pick **different** patches? The matrix counts how many patches two experts share on image 0
(the diagonal is `l`). If the off-diagonal numbers are large, the routers have not specialised (yet).
The paper adds no load-balancing loss for pMoE, so nothing forces them apart.

In [ ]:
k = top_idx.shape[1]
shared = np.array([[len(set(top_idx[0, a]) & set(top_idx[0, b])) for b in range(k)] for a in range(k)])
print(shared)

## 7. MGE, version 1: depth experts trained together
Each expert has its own pooling and class layer, attached after the 64, 128 and 256 channel stages.
The diversity penalty (weight `diversity_weight`) pushes their predicted class distributions apart.
The history has one accuracy per output.

In [ ]:
hist_mge = moe_train.run('mge', seed=0, train_ds=train_ds, val_ds=val_ds, size=SIZE, n_classes=N_CLASSES,
                         epochs=EPOCHS, log_dir=LOG_DIR, diversity_weight=moe_config.DIVERSITY_WEIGHT)
h = hist_mge.history
print([key for key in h if key.startswith('val_')])

for name in ['ensemble', 'expert_0', 'expert_1', 'expert_2']:
    plt.plot(h[f'val_{name}_accuracy'], marker='o', label=name)
plt.title('MGE: validation accuracy per output (expert_0 = 64 ch, expert_2 = 256 ch)')
plt.xlabel('epoch'); plt.legend(); plt.show()

**Check the penalty yourself.** `diversity_penalty_value` is a plain function: it is 1 when two experts give
identical predictions and tends to 0 when they strongly disagree. (The exact formula is a guess, see the README.)

In [ ]:
a = tf.constant([[5., 0, 0], [0, 5., 0]])          # two images, 3 classes, confident scores
print('same predictions     :', float(mge.diversity_penalty_value([a, a])))
print('opposite predictions :', float(mge.diversity_penalty_value([a, a[::-1]])))

## 8. MGE, version 2 (optional): add the experts one at a time
Stage 1 trains one expert, stage 2 two, stage 3 all three, on shared weights.
**Caution:** the paper's exact schedule was not available when this was written, so the order and the epochs are guesses
(see the README).

In [ ]:
hist_gradual = moe_train.run('mge_gradual', seed=0, train_ds=train_ds, val_ds=val_ds, size=SIZE, n_classes=N_CLASSES,
                             epochs=EPOCHS, stage_epochs=STAGE_EPOCHS, log_dir=LOG_DIR)
print('stage of each epoch:', hist_gradual.history['stage'])
plotting.plot_history(hist_gradual)

## 9. The side-by-side comparison
`compare` trains every model on the same data and seeds and prints the parameter count next to the best val macro-F1.
It re-trains everything, so skip it if you only wanted the sections above.

In [ ]:
results = moe_train.compare(train_ds, val_ds, size=SIZE, n_classes=N_CLASSES, seeds=SEEDS,
                            epochs=EPOCHS, log_dir=LOG_DIR)

## 10. Things to try
Pick one, change **one** thing, re-run the section:

* more experts: `n_experts=8` in the `pmoe` run. Do the experts overlap less?
* fewer or more patches kept: `l=2`, or `l=32`. What happens to the F1 and to the picked patches?
* a bigger hidden layer for the pMoE experts: `hidden=128` (default `PMOE_HIDDEN`)
* MGE without the penalty: `diversity_weight=0.0`, then `1.0`. Do the three experts stay different?
* flip the KL direction: `direction='later_vs_earlier'`
* change which expert is "first": `order=(0, 1, 2)`
* set `QUICK = False` and run `compare` on the full data

Defaults live in `utils/moe_project/config.py` (the README has a table of which file and argument to change for each edit).

**Real runs on Colab:** the VM disk disappears. The CSV logs of this notebook go to `history_logs/tutorial/` and are not pushed.
For runs you want to keep, use the GitHub push helper from `MoE_approach.ipynb` (`push_files`).